# String Cleaning

## Overview
Real-world text data arrives with inconsistent casing, extra whitespace, mixed delimiters, embedded symbols, and non-standard formats. SQL string functions let you normalise all of this without leaving the database.

**Core string functions — SQLite and PostgreSQL:**

| Function | Purpose |
|---|---|
| `TRIM(col)` | Remove leading and trailing whitespace |
| `LTRIM(col)` / `RTRIM(col)` | Remove leading / trailing whitespace only |
| `UPPER(col)` / `LOWER(col)` | Convert case |
| `REPLACE(col, old, new)` | Replace all occurrences of a substring |
| `SUBSTR(col, start, length)` | Extract a substring (1-indexed) |
| `LENGTH(col)` | Character count |
| `INSTR(col, substr)` | Position of first occurrence (0 = not found) |
| `||` | String concatenation operator |

**PostgreSQL extras:** `INITCAP()`, `LPAD()`, `RPAD()`, `SPLIT_PART()`, `REGEXP_REPLACE()`, `TRANSLATE()`

**Cleaning order of operations:**
1. TRIM whitespace first — everything else is affected by leading/trailing spaces
2. Normalise case
3. Replace or remove unwanted characters
4. Validate length and format
5. Map to a controlled vocabulary (CASE WHEN)

---

In [1]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(":memory:")
conn.executescript("""
CREATE TABLE intake_raw (record_id INTEGER PRIMARY KEY, patient_ref TEXT, first_name TEXT, last_name TEXT, dob TEXT, sex TEXT, province TEXT, phone TEXT, email TEXT, cost_str TEXT, intake_date TEXT, notes TEXT);
CREATE TABLE lab_raw (lab_id INTEGER PRIMARY KEY, patient_ref TEXT, test_code TEXT, result_txt TEXT, collected TEXT, status TEXT);
CREATE TABLE provider_raw (prov_id INTEGER PRIMARY KEY, name_raw TEXT, dept_code TEXT, start_dt TEXT, salary TEXT);
INSERT INTO intake_raw VALUES
  (1,'P-001','aroha','Ngata','1985-03-12','F','NB','506-555-0101','aroha@mail.com','$3,200.00','2023-01-15','Referred by GP'),
  (2,'P-002','  Liam ','Chen','04/11/1972','Male','NS','902-555-0202','liam@mail.com','1850','15/02/2023',NULL),
  (3,'P-003','FATIMA','AL-RASHID','1990-07-22','female','Ontario','416-555-0303',NULL,'120.00','2023-03-05','Annual checkup'),
  (4,'P-004','James','MacLeod','Jan 30 1955','M','NB',NULL,'james@mail.com','$5,500','18-03-2023','Surgery follow-up'),
  (5,'P-005','sofia','Petrov','2001-09-15','F','BC','604-555-0505','sofia@mail.com','$95.00','2023-04-02',NULL),
  (6,'P-006','Noah','Williams','08/06/1968','m','AB ','780-555-0606','noah@mail.com','780','11/05/2023',NULL),
  (7,'P-007','Mei','Zhang','1995-04-17','F','ON','416-555-0707',NULL,'$2,100.00','22-06-2023','Follow-up required'),
  (8,'P-002','  Liam ','Chen','04/11/1972','Male','NS','902-555-0202','liam@mail.com','1850','15/02/2023',NULL),
  (9,'P-008','ethan','tremblay','01/12/1980',NULL,'QC','514-555-0808','ethan@mail.com','80.00','14-07-2023',NULL),
  (10,'P-009','Priya','Nair','1977-08-25','F','bc',NULL,'priya@mail.com','$1,750.00','01/10/2023',NULL),
  (11,'P-010','Marcus','Okafor','1993-05-19','M','ON','647-555-1010',NULL,'2200','03-11-2023',NULL),
  (12,'P-011','Diana','Flores','14/02/2000','female','NB','506-555-1111','diana@mail.com',NULL,'2023-12-01',NULL),
  (13,NULL,'Unknown',NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,'Incomplete record'),
  (14,'','Test','Record','2023-01-01','X','XX','000-000-0000','test@test.com','-1','2023-01-01','Test entry');
INSERT INTO lab_raw VALUES
  (1,'P-001','HBA1C','7.2 %','2023-03-10','Active'),
  (2,'P-002','HBA1C','9.1%','2023-03-15','active'),
  (3,'P-003','CREAT','88.5umol/L','2023-04-01','ACTIVE'),
  (4,'P-004','CREAT','145 umol/L','2023-04-12','Active'),
  (5,'P-005','HBA1C','5.8 %','2023-05-05',''),
  (6,'P-006','SODIUM','138mmol/L','2023-05-20','Active'),
  (7,'P-007','SODIUM','151 mmol/L','2023-06-01',NULL),
  (8,'P-001','CREAT','72.3 umol/L','2023-06-18','Active'),
  (9,'P-008','HBA1C','6.4%','2023-07-02','Active'),
  (10,'P-009','CREAT','210umol/L','2023-07-15','active');
INSERT INTO provider_raw VALUES
  (1,'MacNeil, Sarah MD','CARD','2018-01-15','$95,000'),
  (2,'Dr. James Wong','ONCO','2019-03-01','88000'),
  (3,'Fatima Osei M.D.','FAM','2017-06-01','$82,500.00'),
  (4,'Larson, Ethan','ORTH','2020-09-15','91000.00'),
  (5,'Sharma, Priya MD','EMRG','2016-11-01','$78,000'),
  (6,'Lucas Petit','RAD','2021-02-28',NULL);
""")
conn.commit()
print("Schema ready.")

Schema ready.


---
## TRIM and case normalisation

In [2]:
print("=== TRIM + UPPER/LOWER: name and province cleaning ===")
q = """
SELECT  record_id,
        first_name,
        TRIM(first_name)                    AS trimmed,
        UPPER(TRIM(first_name))             AS name_upper,
        -- Title case approximation in SQLite (no INITCAP):
        UPPER(SUBSTR(TRIM(first_name),1,1))
            || LOWER(SUBSTR(TRIM(first_name),2)) AS name_title,
        province,
        UPPER(TRIM(province))               AS province_upper
FROM    intake_raw
ORDER BY record_id
"""
print(pd.read_sql(q, conn).to_string(index=False))

print()
print("PostgreSQL INITCAP():")
print("  SELECT INITCAP(LOWER(TRIM(first_name))) FROM intake_raw;")
print("  -- handles multi-word strings: 'AL-RASHID' -> 'Al-Rashid'")

=== TRIM + UPPER/LOWER: name and province cleaning ===
 record_id first_name trimmed name_upper name_title province province_upper
         1      aroha   aroha      AROHA      Aroha       NB             NB
         2      Liam     Liam       LIAM       Liam       NS             NS
         3     FATIMA  FATIMA     FATIMA     Fatima  Ontario        ONTARIO
         4      James   James      JAMES      James       NB             NB
         5      sofia   sofia      SOFIA      Sofia       BC             BC
         6       Noah    Noah       NOAH       Noah      AB              AB
         7        Mei     Mei        MEI        Mei       ON             ON
         8      Liam     Liam       LIAM       Liam       NS             NS
         9      ethan   ethan      ETHAN      Ethan       QC             QC
        10      Priya   Priya      PRIYA      Priya       bc             BC
        11     Marcus  Marcus     MARCUS     Marcus       ON             ON
        12      Diana   Diana    

---
## REPLACE — stripping and substituting characters

In [3]:
# Normalise sex codes and province abbreviations
print("=== REPLACE + CASE WHEN: normalise sex and province values ===")
q = """
SELECT  record_id,
        sex,
        CASE UPPER(TRIM(sex))
            WHEN 'M'      THEN 'M'
            WHEN 'MALE'   THEN 'M'
            WHEN 'F'      THEN 'F'
            WHEN 'FEMALE' THEN 'F'
            ELSE NULL
        END  AS sex_clean,
        province,
        CASE UPPER(TRIM(province))
            WHEN 'NB'      THEN 'NB'
            WHEN 'NS'      THEN 'NS'
            WHEN 'ON'      THEN 'ON'
            WHEN 'ONTARIO' THEN 'ON'
            WHEN 'BC'      THEN 'BC'
            WHEN 'AB'      THEN 'AB'
            WHEN 'QC'      THEN 'QC'
            ELSE NULL
        END  AS province_clean
FROM    intake_raw
ORDER BY record_id
"""
print(pd.read_sql(q, conn).to_string(index=False))

=== REPLACE + CASE WHEN: normalise sex and province values ===
 record_id    sex sex_clean province province_clean
         1      F         F       NB             NB
         2   Male         M       NS             NS
         3 female         F  Ontario             ON
         4      M         M       NB             NB
         5      F         F       BC             BC
         6      m         M      AB              AB
         7      F         F       ON             ON
         8   Male         M       NS             NS
         9   None      None       QC             QC
        10      F         F       bc             BC
        11      M         M       ON             ON
        12 female         F       NB             NB
        13   None      None     None           None
        14      X      None       XX           None


---
## Extracting substrings — parsing structured text fields

In [4]:
# Extract numeric part from lab result_txt: "7.2 %", "88.5umol/L", "145 umol/L"
# Strategy: take characters up to the first non-numeric/dot character
print("=== Extract numeric value from lab result_txt ===")
q = """
SELECT  lab_id,
        patient_ref,
        test_code,
        result_txt,
        -- Remove unit suffixes by keeping only digits and decimal point
        -- SQLite has no regex replace, so use TRIM with characters
        CAST(TRIM(REPLACE(REPLACE(REPLACE(REPLACE(
            result_txt,'%',''),'umol/L',''),'mmol/L',''),' ',''))
        AS REAL)  AS result_numeric,
        -- Extract unit
        CASE
            WHEN INSTR(result_txt,'%')      > 0 THEN '%'
            WHEN INSTR(result_txt,'umol/L') > 0 THEN 'umol/L'
            WHEN INSTR(result_txt,'mmol/L') > 0 THEN 'mmol/L'
            ELSE 'unknown'
        END        AS unit_extracted
FROM    lab_raw
ORDER BY lab_id
"""
print(pd.read_sql(q, conn).to_string(index=False))

=== Extract numeric value from lab result_txt ===
 lab_id patient_ref test_code  result_txt  result_numeric unit_extracted
      1       P-001     HBA1C       7.2 %             7.2              %
      2       P-002     HBA1C        9.1%             9.1              %
      3       P-003     CREAT  88.5umol/L            88.5         umol/L
      4       P-004     CREAT  145 umol/L           145.0         umol/L
      5       P-005     HBA1C       5.8 %             5.8              %
      6       P-006    SODIUM   138mmol/L           138.0         mmol/L
      7       P-007    SODIUM  151 mmol/L           151.0         mmol/L
      8       P-001     CREAT 72.3 umol/L            72.3         umol/L
      9       P-008     HBA1C        6.4%             6.4              %
     10       P-009     CREAT   210umol/L           210.0         umol/L


---
## Normalising provider names — SUBSTR and INSTR

In [5]:
# provider_raw has: "MacNeil, Sarah MD", "Dr. James Wong", "Fatima Osei M.D."
# Goal: extract a clean surname and first name where possible
print("=== Provider name parsing ===")
q = """
SELECT  prov_id,
        name_raw,
        -- Detect "Lastname, Firstname" format by looking for comma
        CASE WHEN INSTR(name_raw, ',') > 0
            THEN TRIM(SUBSTR(name_raw, 1, INSTR(name_raw,',')-1))
            ELSE NULL
        END  AS surname_if_comma_format,
        -- Strip titles and credentials
        TRIM(REPLACE(REPLACE(REPLACE(REPLACE(
            name_raw,'Dr. ',''),' MD',''),' M.D.',''),'Dr.',''))
            AS name_no_titles,
        LENGTH(TRIM(name_raw))   AS name_length
FROM    provider_raw
ORDER BY prov_id
"""
print(pd.read_sql(q, conn).to_string(index=False))

print()
print("PostgreSQL REGEXP_REPLACE for cleaner extraction:")
print("  SELECT REGEXP_REPLACE(name_raw, '(Dr\\.|M\\.?D\\.?|MD)', '', 'g') FROM provider_raw;")
print("  SELECT SPLIT_PART(name_raw, ', ', 1) AS surname FROM provider_raw;")

conn.close()

=== Provider name parsing ===
 prov_id          name_raw surname_if_comma_format name_no_titles  name_length
       1 MacNeil, Sarah MD                 MacNeil MacNeil, Sarah           17
       2    Dr. James Wong                    None     James Wong           14
       3  Fatima Osei M.D.                    None    Fatima Osei           16
       4     Larson, Ethan                  Larson  Larson, Ethan           13
       5  Sharma, Priya MD                  Sharma  Sharma, Priya           16
       6       Lucas Petit                    None    Lucas Petit           11

PostgreSQL REGEXP_REPLACE for cleaner extraction:
  SELECT REGEXP_REPLACE(name_raw, '(Dr\.|M\.?D\.?|MD)', '', 'g') FROM provider_raw;
  SELECT SPLIT_PART(name_raw, ', ', 1) AS surname FROM provider_raw;


---
## Common Pitfalls

**1. Forgetting to TRIM before any other string operation**
`UPPER('  Liam ')` returns `'  LIAM '` — the spaces are still there. Always TRIM first: `UPPER(TRIM(col))`. A leading space makes `'ON'` and `' ON'` unequal in every comparison and GROUP BY.

**2. REPLACE is case-sensitive in SQLite**
`REPLACE(col, 'male', '')` does not remove `'Male'` or `'MALE'`. Normalise case before REPLACE: `REPLACE(LOWER(TRIM(col)), 'male', '')`.

**3. Nested REPLACE chains become unreadable and fragile**
`REPLACE(REPLACE(REPLACE(REPLACE(col,'$',''),',',''),' ',''),'%',''))` is hard to audit. For complex cleaning, consider pulling data into pandas and using `str.replace()` or regex, then writing back. Document what each REPLACE step removes.

**4. SUBSTR is 1-indexed in SQLite and PostgreSQL**
`SUBSTR(name, 1, 5)` returns the first 5 characters starting from position 1. `SUBSTR(name, 0, 5)` in SQLite starts from position 0 (treated as 1) — avoid position 0 for portability.

**5. INSTR returns 0 (not NULL) when the substring is not found**
`INSTR(col, ',')` returns 0 if there is no comma. `SUBSTR(name, 1, INSTR(name,',')-1)` then returns `SUBSTR(name, 1, -1)` which is an empty string in SQLite. Always guard: `CASE WHEN INSTR(name,',') > 0 THEN ... END`.

**6. String length in bytes vs characters for multi-byte Unicode**
`LENGTH()` in SQLite returns character count for UTF-8 text (correct for most purposes). `OCTET_LENGTH()` in PostgreSQL returns byte count — relevant for BLOB or binary columns. For healthcare data with accented characters (e.g. `'Tremblay'`, `'Al-Rashid'`), verify your database encoding is UTF-8.


---
*sql_methods_library - Samantha McGarrigle*